In [1]:
%pip install -q "qiskit==2.2.3" "qiskit-optimization==0.7.0" "koreanize-matplotlib==0.1.1"

import sys
from importlib.metadata import version
print("Python:", sys.version.split()[0])
for name in ["qiskit", "qiskit-optimization", "numpy", "scipy", "torch", "matplotlib"]:
    print(name, version(name))
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import COBYLA
print("설치 및 import 확인 완료")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Python: 3.13.18
qiskit 2.2.3
qiskit-optimization 0.7.0
numpy 2.1.3
scipy 1.16.3
torch 2.11.0+cpu
matplotlib 3.10.0
설치 및 import 확인 완료


In [5]:
from itertools import product
import numpy as np
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer

def solve_checked(problem, reps=2, maxiter=100, seed=42):
    # 원문제의 제약을 유지한 채 내부 QUBO 변환과 역변환을 맡깁니다.
    qaoa = QAOA(
        sampler=StatevectorSampler(default_shots=4096, seed=seed),
        optimizer=COBYLA(maxiter=maxiter), reps=reps,
        initial_point=np.random.default_rng(seed).uniform(0.1, 1.0, 2*reps),
    )
    result = MinimumEigenOptimizer(qaoa).solve(problem)
    if result.x is None or not problem.is_feasible(result.x):
        raise RuntimeError("유효한 선택을 얻지 못했습니다. 샷·초깃값·탐색 예산을 바꿔 재실행하세요.")
    assert np.isclose(np.sum(result.x), 1)
    assert np.all(np.isclose(result.x, 0) | np.isclose(result.x, 1))
    assert np.isclose(result.fval, problem.objective.evaluate(result.x))
    feasible = [np.array(x) for x in product([0,1], repeat=problem.get_num_vars())
                if problem.is_feasible(np.array(x))]
    exact = min(float(problem.objective.evaluate(x)) for x in feasible)
    valid_mass = sum(float(s.probability) for s in result.samples if problem.is_feasible(s.x))
    best_mass = sum(float(s.probability) for s in result.samples
                    if problem.is_feasible(s.x) and np.isclose(s.fval, exact))
    gap = float(result.fval-exact)
    assert gap >= -1e-6 and np.isfinite(result.fval)
    assert np.isclose(sum(float(s.probability) for s in result.samples),1)
    print(f"정확한 최솟값={exact:.6f}, QAOA 후보 비용={result.fval:.6f}, 차이={gap:.6f}")
    print(f"유효 후보 관측 비율={valid_mass:.4f}, 최적 후보 관측 비율={best_mass:.4f}")
    return result

# ============================================
# QAOA 깊이(reps) (서울역 → 강남역)
# ============================================

import time

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
import numpy as np

# -------------------------------------------------------
# (1) 경로 데이터 설정
# -------------------------------------------------------
routes = {
    "A_지하철": {"time": 32, "congestion": 2},
    "B_버스": {"time": 45, "congestion": 4},
    "C_자전거+지하철": {"time": 38, "congestion": 1},
    "D_택시": {"time": 28, "congestion": 4},
    "E_도보+9호선": {"time": 40, "congestion": 1},
}

for k, v in routes.items():
    v["cost"] = v["time"] + 5 * v["congestion"]

# -------------------------------------------------------
# (2) 문제 정의
# -------------------------------------------------------
problem = QuadraticProgram()
for name in routes.keys():
    problem.binary_var(name)
problem.linear_constraint({name: 1 for name in routes.keys()}, '==', 1)
problem.minimize(linear={name: v["cost"] for name, v in routes.items()})



# -------------------------------------------------------
# (3) QAOA 깊이별 비교
# -------------------------------------------------------
reps_list = [1, 3]
results = []

for reps in reps_list:
    start = time.time()
    qaoa_reps, qaoa_maxiter = reps, 100

    result = solve_checked(problem, reps=qaoa_reps, maxiter=qaoa_maxiter)
    elapsed = time.time() - start
    selected = [k for k, v in zip(routes.keys(), result.x) if v == 1][0]
    results.append({
        "reps": reps,
        "route": selected,
        "cost": routes[selected]["cost"],
        "fval": result.fval,
        "time": elapsed
    })

# -------------------------------------------------------
# (4) 결과 비교 출력
# -------------------------------------------------------
print("============================================")
print("QAOA 깊이(reps)별 결과 비교")
print("============================================")
for r in results:
    print(f"[reps={r['reps']}]")
    print(f"  선택된 경로: {r['route']}")
    print(f"  총 이동비용: {r['cost']}")
    print(f"  fval(최소비용값): {r['fval']:.4f}")
    print(f"  계산시간: {r['time']:.3f}초")
    print("--------------------------------------------")

정확한 최솟값=42.000000, QAOA 후보 비용=42.000000, 차이=0.000000
유효 후보 관측 비율=0.4148, 최적 후보 관측 비율=0.0054
정확한 최솟값=42.000000, QAOA 후보 비용=42.000000, 차이=0.000000
유효 후보 관측 비율=0.3723, 최적 후보 관측 비율=0.0469
QAOA 깊이(reps)별 결과 비교
[reps=1]
  선택된 경로: A_지하철
  총 이동비용: 42
  fval(최소비용값): 42.0000
  계산시간: 0.588초
--------------------------------------------
[reps=3]
  선택된 경로: A_지하철
  총 이동비용: 42
  fval(최소비용값): 42.0000
  계산시간: 1.404초
--------------------------------------------
